# เทรน LSTM จำแนกคำภาษามือไทยใน Google Colab

ค่าเริ่มต้นเทรน **TSL‑51 ครบ 52 คลาส** (51 รูปท่าและคลาส `null_act`) จาก landmark ของล่าม 2 คน ใช้คลิปต้นฉบับจากชุด `user_sign` ของผู้เซ็นอีกคนสำหรับทดสอบ เมื่อจบจะได้โมเดล TorchScript, โค้ด Python สำหรับแปลง landmark และทำนาย, รายชื่อคลาส, ตารางผลรายคลาส และไฟล์ ZIP

1. อัปโหลดไฟล์ `.ipynb` ที่ [colab.research.google.com](https://colab.research.google.com/) เลือก **Runtime → Change runtime type → GPU** แล้วรัน **Run all**
2. เซลล์ดาวน์โหลดจะจัดการไฟล์ TSL‑51 ที่ต้องใช้ให้ทั้งหมด ตั้ง `DOWNLOAD_TSL51 = "all"` หากต้องการวิดีโอ คลิปประโยค และชุดจากเว็บที่ Hugging Face เผยแพร่ด้วย ชุดเต็มประมาณ 2.56 GB และควรเผื่อพื้นที่อีกหลาย GB สำหรับ cache
3. หากต้องการเก็บผลข้าม Colab session ให้ตั้ง `SAVE_TO_DRIVE = True` เซลล์นี้จะขอให้เชื่อม Google Drive หรือใช้เซลล์ดาวน์โหลด ZIP ท้าย Notebook

ขอบเขตของโมเดลคือคลิปที่ตัดมาแล้วคลิปละหนึ่งรูปท่า ใช้ `sign_id` ของเจ้าของชุดเพื่อเก็บท่าที่ต่างกันไว้ เช่น `ฉัน_var_1` กับ `ฉัน_var_2` รูปคำบนจอและความสุภาพตามผู้รับสารให้กำหนดในบทเรียนของเกมตามบริบท

ถ้าดาวน์โหลดเอง ให้เก็บไฟล์ตามโครงสร้างเดิมจากหน้า Files ของ Hugging Face แล้วตั้ง `DOWNLOAD_TSL51 = "local"` และกำหนด `TSL51_DIR` ให้ชี้ไปยังโฟลเดอร์ที่มี `metadata/` และ `landmarks/`

TSL‑51 ระบุสัญญาอนุญาต CC BY‑NC‑SA 4.0 และระบุข้อจำกัดสิทธิ์เพิ่มเติมสำหรับข้อมูลจากบุคคลที่สาม ค่าเริ่มต้นใช้ชุดล่ามหลักเพื่อวิจัยและชุดทดสอบของผู้จัดทำ หากจะนำไปใช้เชิงพาณิชย์ให้ตรวจสิทธิ์และขออนุญาตเจ้าของสิทธิ์ก่อน [รายละเอียดและที่มา TSL‑51](https://huggingface.co/datasets/Namonpas/thai-sign-language-tsl51/blob/main/README.md)

## 1. แหล่งข้อมูลและวิธีเข้าถึง

| ชุดข้อมูล | วิธีรับข้อมูล | บทบาทใน Notebook นี้ |
|---|---|---|
| [TSL‑51](https://huggingface.co/datasets/Namonpas/thai-sign-language-tsl51/tree/main) | โหลดอัตโนมัติจาก Hugging Face เลือก `"training"`, `"all"` หรือ `"local"` | เทรนคำเดี่ยวจาก landmark ของล่ามหลักและประเมินบน `user_sign` |
| [TSL‑ONE‑S](https://tsl-one.github.io/) | เซลล์เสริมโหลด [TSL‑ONE‑Pose.7z](https://github.com/TSL-ONE/TSL-ONE.github.io/blob/62ff4924873fbb20b8691bc36591dfa72c0b4f9a/data/TSL-ONE-Pose.7z) ได้ ส่วน MP4 ขอผ่านฟอร์มของโครงการ | keypoint `.npy` ใช้ชุดจุดและป้ายกำกับของตัวเอง หากได้ MP4 มาแล้วสามารถสกัด landmark ให้ตรงสเปกตามหัวข้อเพิ่มข้อมูล |
| [One‑Stage‑TFS](https://data.mendeley.com/datasets/rknd3wbz42/1) | ดาวน์โหลดด้วยปุ่ม **Download All** ที่ Mendeley แล้วอัปโหลดถ้าต้องการใช้ใน Colab | ภาพนิ่ง JPG/XML สำหรับสะกดนิ้ว |
| [ATFS](https://github.com/beebrain/ATFS_Thai_Finger_Spelling) | เซลล์เสริมดาวน์โหลด repository ซึ่งมีโค้ดและข้อมูลประกอบ ตรวจไฟล์วิดีโอดิบและสิทธิ์กับผู้จัดทำหากต้องใช้ | งานถอดลำดับสะกดนิ้ว |
| [ThaiSignVis](https://www.kaggle.com/datasets/thanawuttimpitak/thaisignvis) | เซลล์เสริมใช้ Kaggle Hub วิดีโอจริงอยู่ที่ Kaggle ตาม [หน้า Hugging Face](https://huggingface.co/datasets/TOMTUN/ThaiSignVis) | วิดีโอต่อเนื่องพร้อม transcript ขนาดใหญ่ |
| [DTFS](https://khoon.msu.ac.th/_dir/fulltext/2024/04/Siriwiwat_Lata23.pdf), [JAIST TFS](https://dspace.jaist.ac.jp/dspace/bitstream/10119/19331/3/summary.pdf), [Thai Sign Digits](https://www.scitepress.org/Papers/2023/116437/116437.pdf) | เอกสารอธิบายชุดข้อมูล ให้ติดต่อผู้วิจัยเพื่อขอไฟล์และเงื่อนไข | ชุดสะกดนิ้วและตัวเลข |
| [TH‑SL](https://www.th-sl.com/), [พจนานุกรมราชบัณฑิตยสภา](http://164.115.33.116/manual.html), หลักสูตรและคลังคำศัพท์ในรายงาน | เปิดดูแหล่งอ้างอิง หรือติดต่อเจ้าของถ้าจะใช้ไฟล์สาธิต | ตรวจรูปท่าและคำในบทเรียน |
| [โครงการ AIT](https://ait.ac.th/project/development-of-sign-language-intelligence-and-its-applications/) | ติดตามช่องทางเผยแพร่จากโครงการ | แหล่งข้อมูลสำหรับงานต่อยอด |
| ชุดอายุและความสุภาพตามรายงาน | เก็บคลิปเอง หรือขอข้อมูลที่มีคำและรูปท่าตรงกัน | นำเข้า landmark ผ่าน manifest ได้ เช่น เมื่อได้ตัวอย่างคำ **ยินดี** ที่ตรวจสอบรูปท่าแล้ว |

คลาสสวัสดีและขอโทษของ TSL‑51 มีคำกำกับว่าใช้กับคนอายุเท่ากันหรือน้อยกว่า คำอย่าง **ยินดี** และ **ขออนุญาต** ไม่มีคลาสใน metadata เวอร์ชันที่ใช้ [รายการรูปท่า TSL‑51](https://huggingface.co/datasets/Namonpas/thai-sign-language-tsl51/blob/main/README.md)

In [ ]:
from pathlib import Path

WORK_DIR = Path("/content/tsl_lstm")
SAVE_TO_DRIVE = False
DOWNLOAD_TSL51 = "training"     # "all" โหลดทั้ง repository, "local" ใช้ไฟล์เดิม
TSL51_DIR = WORK_DIR / "tsl51"   # เปลี่ยนเป็นโฟลเดอร์ใน Drive เมื่อตั้ง "local"
INCLUDE_WEB_SCRAPED = False      # เปิดเฉพาะเมื่อสิทธิ์อนุญาต ใช้ใน train เท่านั้น
EXTRA_MANIFEST = None            # ตัวอย่าง Path("/content/drive/MyDrive/tsl_extra/manifest.csv")

SEED = 42
FRAMES = 48
VAL_FRACTION = 0.20
EPOCHS = 50
BATCH_SIZE = 128
LEARNING_RATE = 0.001
PATIENCE = 9

assert FRAMES >= 2 and EPOCHS > 0 and BATCH_SIZE > 0 and PATIENCE > 0
assert LEARNING_RATE > 0

RUN_NAME = "tsl51_lstm"
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUTPUT_DIR = Path("/content/drive/MyDrive/Thai_Sign_Language_Game") / RUN_NAME
else:
    OUTPUT_DIR = WORK_DIR / RUN_NAME

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
WORK_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("huggingface_hub") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub>=0.34,<2"])

from huggingface_hub import snapshot_download

REPO = "Namonpas/thai-sign-language-tsl51"
REVISION = "6d3dde16db90e5cc10c6fd5e8a36e3be18a68df5"
EXPERT_ZIPS = ["expert_primary_02.zip", "expert_primary_03.zip"]
if INCLUDE_WEB_SCRAPED:
    EXPERT_ZIPS.append("expert_scraped.zip")

assert DOWNLOAD_TSL51 in {"training", "all", "local"}
if DOWNLOAD_TSL51 != "local":
    training_paths = [
        "README.md", "metadata/expert_metadata.csv", "metadata/user_sign_metadata.csv",
        "landmarks/user_sign/*.csv",
        *[f"landmarks/{name}" for name in EXPERT_ZIPS],
    ]
    snapshot_download(
        REPO, repo_type="dataset", revision=REVISION, local_dir=TSL51_DIR,
        allow_patterns=training_paths if DOWNLOAD_TSL51 == "training" else None,
        max_workers=4,
    )

for rel in [
    "metadata/expert_metadata.csv", "metadata/user_sign_metadata.csv",
    *[f"landmarks/{name}" for name in EXPERT_ZIPS],
]:
    if not (TSL51_DIR / rel).is_file():
        raise FileNotFoundError(f"ไม่พบไฟล์ {TSL51_DIR / rel}")
print("TSL-51:", TSL51_DIR, "revision:", REVISION if DOWNLOAD_TSL51 != "local" else "local")

### เพิ่มคลิปของโปรเจกต์ (ถ้ามี)

ตั้ง `EXTRA_MANIFEST` ให้ชี้ไปที่ CSV แบบ UTF‑8 เปิด `SAVE_TO_DRIVE` เพื่อเชื่อม Drive หากใช้ path ใต้ `/content/drive` ตัวอย่างหัวตารางและข้อมูล

```csv
landmark_path,sign_id,sign_clean,signer_id,split
clips/p01_yindee_01.csv,ยินดี_var_1,ยินดี,p01,train
clips/p02_yindee_01.csv,ยินดี_var_1,ยินดี,p02,val
clips/p03_yindee_01.csv,ยินดี_var_1,ยินดี,p03,test
```

ตัวอย่างนี้แสดงรูปแบบคอลัมน์ ให้ใช้หลายคลิปจริงต่อคำและหลายผู้เซ็น กำหนดผู้เซ็นแต่ละคนให้อยู่ split เดียว และอย่านำผู้เซ็นในชุดทดสอบ TSL‑51 มาใส่ในชุดเสริม คำใหม่ทุกคำต้องมีคลิปในทั้งสาม split ถ้าเป็นรูปท่าเดิมของ TSL‑51 ให้ใช้ `sign_id` และ `sign_clean` ที่ปรากฏใน metadata ตรงตัว

ถ้าได้รับ MP4 มา ให้ตัดทีละท่าแล้วสกัด MediaPipe Holistic แบบเดียวกับ TSL‑51 ลง CSV หนึ่งไฟล์ต่อคลิป ใช้ชื่อและความหมายของพิกัดตาม `COORDINATE_COLUMNS` ด้านล่าง ใช้ค่าเว้นว่างหรือ NaN เมื่อจับจุดไม่ได้ และบันทึก `frame` กับ `t_ms` หากมี ชุด NPY จาก TSL‑ONE‑S ต้องมีการแปลงจุดและจับคู่ gloss ที่ตรวจสอบแล้วก่อนใช้ [สเปก landmark ของ TSL‑51](https://huggingface.co/datasets/Namonpas/thai-sign-language-tsl51/blob/main/README.md#landmark-format)

In [ ]:
%%writefile /content/tsl_preprocess.py
import csv

import numpy as np


ARM_POINTS = ("l_shoulder", "r_shoulder", "l_elbow", "r_elbow", "l_wrist", "r_wrist")
FACE_POINTS = ("lbrow_outer", "lbrow_inner", "rbrow_inner", "rbrow_outer", "mouth_right", "mouth_left")
COORDINATE_COLUMNS = (
    [f"{side}_{axis}{point}" for side in ("lh", "rh") for point in range(21) for axis in "xyz"]
    + [f"{point}_{axis}" for point in ARM_POINTS + FACE_POINTS for axis in "xyz"]
)
N_POINTS = len(COORDINATE_COLUMNS) // 3
INPUT_FEATURES = 4 * N_POINTS


def landmarks_to_tensor(coordinates, frames, time_ms=None):
    """Signed interval -> float32 array [frames, 216] in the trained feature order."""
    points = np.asarray(coordinates, dtype=np.float32).reshape(-1, N_POINTS, 3)
    if len(points) < 2:
        raise ValueError("Need at least two frames")
    if frames < 2:
        raise ValueError("frames must be at least two")

    valid = np.isfinite(points).all(axis=-1)
    if not valid.any():
        raise ValueError("No detected landmarks")
    points = np.where(valid[..., None], points, 0.0)

    left, right = points[:, 42, :2], points[:, 43, :2]
    widths = np.linalg.norm(left - right, axis=-1)
    anchors = valid[:, 42] & valid[:, 43] & (widths > 0.025)
    if anchors.any():
        indexes = np.arange(len(points))
        center = (left + right) / 2.0
        center = np.column_stack([
            np.interp(indexes, indexes[anchors], center[anchors, axis]) for axis in range(2)
        ])
        points[:, :, :2] -= center[:, None, :]
        points /= np.median(widths[anchors])
    else:
        points[:, :, :2] -= 0.5

    clock = np.arange(len(points), dtype=np.float64)
    if time_ms is not None:
        stamps = np.asarray(time_ms, dtype=np.float64)
        if stamps.shape == clock.shape and np.isfinite(stamps).all() and (np.diff(stamps) > 0).all():
            clock = stamps

    target = np.linspace(clock[0], clock[-1], frames)
    hi = np.searchsorted(clock, target, side="right").clip(1, len(points) - 1)
    lo = hi - 1
    alpha = ((target - clock[lo]) / (clock[hi] - clock[lo]))[:, None, None]
    nearest = np.where(alpha[:, 0, 0] < 0.5, lo, hi)
    both = valid[lo] & valid[hi]
    interpolated = points[lo] + alpha * (points[hi] - points[lo])
    selected = np.where(both[..., None], interpolated, points[nearest])
    selected = np.where(valid[nearest, :, None], selected, 0.0)
    return np.concatenate(
        [selected.reshape(frames, -1), valid[nearest]], axis=-1,
    ).astype(np.float32)


def read_landmark_csv(file, frames):
    """Read an open UTF-8/UTF-8-SIG text stream in the TSL-51 CSV format."""
    reader = csv.reader(file)
    header = next(reader, [])
    index = {name.strip(): i for i, name in enumerate(header)}
    missing = [column for column in COORDINATE_COLUMNS if column not in index]
    if missing:
        raise ValueError(f"Missing landmark columns: {missing[:8]} ({len(missing)} total)")
    columns = [index[name] for name in COORDINATE_COLUMNS]
    time_column = index.get("t_ms")
    values, stamps = [], []
    for row in reader:
        if not row:
            continue
        if len(row) != len(header):
            raise ValueError(f"Wrong column count at CSV line {reader.line_num}")
        values.append([row[i].strip() or "nan" for i in columns])
        if time_column is not None:
            stamps.append(row[time_column].strip() or "nan")
    return landmarks_to_tensor(values, frames, stamps if time_column is not None else None)

## 2. แบ่งคลิปและเตรียม landmark

- แบ่งข้อมูลล่ามตาม `source` และ `original_video_id` โดยสุ่มคลิปต้นฉบับแต่ละคลาสเข้า validation ข้อมูล augmentation มาจากฝั่ง train เท่านั้น
- เก็บ `user_sign` ทุกคลิปไว้ทดสอบหลังเลือก checkpoint และเกณฑ์ความมั่นใจแล้ว ชุดนี้วัดผลกับผู้เซ็นเพิ่มอีกหนึ่งคน
- แต่ละคลิปถูกปรับเป็น 48 เฟรมต่อเนื่อง คำนวณตำแหน่งเทียบกับไหล่เมื่อพบจุด และเพิ่ม mask 54 ค่าเพื่อระบุ landmark ที่ตรวจพบ รวม 216 features ต่อเฟรม ใช้การแปลงชุดเดียวกันตอนเทรนและทำนาย

In [ ]:
import csv
import hashlib
import importlib
import io
import json
import math
import os
import posixpath
import shutil
import unicodedata
from collections import Counter, defaultdict
from contextlib import ExitStack
from pathlib import PurePosixPath
from zipfile import ZipFile

import numpy as np
from tqdm.auto import tqdm

if "/content" not in sys.path:
    sys.path.insert(0, "/content")
import tsl_preprocess as preprocess
preprocess = importlib.reload(preprocess)


def read_metadata(path, required):
    with path.open(encoding="utf-8-sig", newline="") as stream:
        reader = csv.DictReader(stream)
        missing = set(required) - set(reader.fieldnames or ())
        if missing:
            raise ValueError(f"{path}: missing columns {sorted(missing)}")
        return [{k: (row[k] or "").strip() for k in required} for row in reader]


def make_splits(root, include_web, val_fraction, seed, extra_manifest=None):
    if not 0 < val_fraction < 1:
        raise ValueError("VAL_FRACTION must be between 0 and 1")

    common = ("video_id", "source", "sign_id", "sign_clean", "landmark_path", "extraction_status")
    expert = read_metadata(root / "metadata/expert_metadata.csv", (
        *common, "source_group", "original_video_id", "is_augmented",
    ))
    user = read_metadata(root / "metadata/user_sign_metadata.csv", common)
    if any(r["extraction_status"] != "ok" for r in expert + user):
        print("Metadata rows with extraction errors will be excluded.")
    expert = [r for r in expert if r["extraction_status"] == "ok"]
    user = [r for r in user if r["extraction_status"] == "ok"]

    original_by_class = defaultdict(list)
    group_by_id = {}
    for row in expert:
        if row["source_group"] not in {"expert-primary", "web-scraped"}:
            raise ValueError(f"Unknown expert source group: {row['source_group']}")
        if row["source_group"] == "web-scraped" and not include_web:
            continue
        if row["is_augmented"].lower() not in {"true", "false"} or not row["original_video_id"]:
            raise ValueError(f"Invalid augmentation metadata: {row['video_id']}")
        group = (row["source"], row["original_video_id"])
        group_by_id.setdefault(group, []).append(row)

    for group, variants in group_by_id.items():
        originals = [r for r in variants if r["is_augmented"].lower() == "false"]
        if len(originals) != 1 or len({r["sign_id"] for r in variants}) != 1:
            raise ValueError(f"Invalid original-clip group: {group}")
        if originals[0]["source_group"] == "expert-primary":
            original_by_class[originals[0]["sign_id"]].append(group)

    if len(original_by_class) != 52 or "null_act" not in original_by_class:
        raise ValueError(f"Expected 52 TSL-51 primary classes: got {len(original_by_class)}")
    if {r["sign_id"] for r in user} != set(original_by_class):
        raise ValueError("TSL-51 user metadata has unexpected or missing classes")

    rng = np.random.default_rng(seed)
    val_groups = set()
    for label, groups in sorted(original_by_class.items()):
        groups.sort()
        if len(groups) < 2:
            raise ValueError(f"{label}: need two or more originals")
        count = min(len(groups) - 1, max(1, round(len(groups) * val_fraction)))
        val_groups.update(groups[i] for i in rng.permutation(len(groups))[:count])

    rows = []
    for group, variants in sorted(group_by_id.items()):
        for row in variants:
            augmented = row["is_augmented"].lower() == "true"
            if group in val_groups and augmented:
                continue
            rows.append({
                **row, "storage": "expert", "split": "val" if group in val_groups else "train",
                "group": "/".join(group),
            })
    for row in user:
        rows.append({**row, "storage": "user", "split": "test", "group": row["source"]})

    if extra_manifest is not None:
        extra_manifest = Path(extra_manifest)
        extra = read_metadata(extra_manifest, ("landmark_path", "sign_id", "sign_clean", "signer_id", "split"))
        if not extra:
            raise ValueError("EXTRA_MANIFEST has no clips")
        split_by_signer = {}
        used_paths = set()
        for row in extra:
            signer, split = row["signer_id"], row["split"].lower()
            if not all(row.values()) or split not in {"train", "val", "test"}:
                raise ValueError(f"Invalid extra-manifest row: {row}")
            if split_by_signer.setdefault(signer, split) != split:
                raise ValueError(f"Signer {signer} appears in more than one split")
            path = (extra_manifest.parent / row["landmark_path"]).resolve(strict=True)
            if path in used_paths or path.is_relative_to(root.resolve()):
                raise ValueError(f"Invalid or repeated extra clip: {path}")
            used_paths.add(path)
            rows.append({
                **row, "landmark_path": str(path), "storage": "extra", "split": split,
                "source": "project_extra", "group": signer, "video_id": path.stem,
            })

    names = {}
    for row in rows:
        label = row["sign_id"]
        name = "ไม่พบรูปท่า" if label == "null_act" else row["sign_clean"]
        if not label or not name or names.setdefault(label, name) != name:
            raise ValueError(f"Inconsistent Thai name for {label}: {name}")
    return rows, names

In [ ]:
def zip_key(path):
    path = posixpath.normpath(path.replace("\\", "/"))
    return unicodedata.normalize("NFC", PurePosixPath(path).name)


def file_stamp(path):
    stat = path.stat()
    return [str(path), stat.st_size, stat.st_mtime_ns]


def build_feature_cache(rows, root, archives, frames, cache_dir):
    # Metadata paths use earlier folder names for the primary ZIPs.
    # Match the unique normalized CSV basename across archives.
    files = [root / "metadata" / name for name in ("expert_metadata.csv", "user_sign_metadata.csv")]
    files.extend(root / "landmarks" / archive for archive in archives)
    files.extend(root / r["landmark_path"] for r in rows if r["storage"] == "user")
    files.extend(Path(r["landmark_path"]) for r in rows if r["storage"] == "extra")
    fingerprint = hashlib.sha256(json.dumps([
        hashlib.sha256(Path(preprocess.__file__).read_bytes()).hexdigest(),
        frames, [file_stamp(path) for path in files], rows,
    ], ensure_ascii=False, sort_keys=True).encode("utf-8")).hexdigest()[:20]

    cache_dir.mkdir(exist_ok=True, parents=True)
    array_path = cache_dir / f"{fingerprint}.npy"
    audit_path = cache_dir / f"{fingerprint}_audit.json"
    if array_path.is_file() and audit_path.is_file():
        return array_path, json.loads(audit_path.read_text(encoding="utf-8"))

    shape = (len(rows), frames, preprocess.INPUT_FEATURES)
    required = math.prod(shape) * np.dtype(np.float32).itemsize
    if shutil.disk_usage(cache_dir).free < required + 100 * 1024**2:
        raise OSError(f"Feature cache needs {required / 1024**3:.2f} GiB of free space")

    temp_path = array_path.with_suffix(".partial")
    with ExitStack() as stack:
        index = defaultdict(list)
        for archive in archives:
            opened = stack.enter_context(ZipFile(root / "landmarks" / archive))
            for member in opened.infolist():
                if not member.is_dir() and member.filename.lower().endswith(".csv"):
                    index[zip_key(member.filename)].append((opened, member))
        sources = []
        for row in rows:
            path = row["landmark_path"]
            if row["storage"] == "expert":
                matches = index.get(zip_key(path), [])
                if len(matches) != 1:
                    raise ValueError(f"Expected one ZIP member for {path}; found {len(matches)}")
                sources.append(matches[0])
            else:
                path = root / path if row["storage"] == "user" else Path(path)
                if not path.is_file():
                    raise FileNotFoundError(path)
                sources.append(path)

        features = np.lib.format.open_memmap(temp_path, mode="w+", dtype=np.float32, shape=shape)
        rejected = []
        try:
            for i, source in enumerate(tqdm(sources, desc="แปลง landmark")):
                if isinstance(source, tuple):
                    archive, member = source
                    stream = io.TextIOWrapper(archive.open(member), encoding="utf-8-sig", newline="")
                else:
                    stream = source.open(encoding="utf-8-sig", newline="")
                with stream:
                    try:
                        features[i] = preprocess.read_landmark_csv(stream, frames)
                    except ValueError as exc:
                        reason = str(exc)
                        if reason not in {"Need at least two frames", "No detected landmarks"}:
                            raise ValueError(f"{rows[i]['landmark_path']}: {reason}") from exc
                        rejected.append({"index": i, "reason": reason})
            features.flush()
        finally:
            del features

    os.replace(temp_path, array_path)
    temporary_audit = audit_path.with_suffix(".partial")
    temporary_audit.write_text(json.dumps(rejected, ensure_ascii=False), encoding="utf-8")
    os.replace(temporary_audit, audit_path)
    return array_path, rejected


rows, thai_names = make_splits(TSL51_DIR, INCLUDE_WEB_SCRAPED, VAL_FRACTION, SEED, EXTRA_MANIFEST)
feature_path, rejected = build_feature_cache(rows, TSL51_DIR, EXPERT_ZIPS, FRAMES, WORK_DIR / "cache")
rejected_by_index = {r["index"]: r["reason"] for r in rejected}
features = np.load(feature_path, mmap_mode="c")
assert features.shape == (len(rows), FRAMES, preprocess.INPUT_FEATURES)

labels = sorted(thai_names)
label_index = {label: i for i, label in enumerate(labels)}
targets = np.array([label_index[r["sign_id"]] for r in rows], dtype=np.int64)
split_indices = {
    split: np.array([
        i for i, r in enumerate(rows) if r["split"] == split and i not in rejected_by_index
    ], dtype=np.int64)
    for split in ("train", "val", "test")
}
for split, indices in split_indices.items():
    missing_labels = set(labels) - {rows[i]["sign_id"] for i in indices}
    if missing_labels:
        raise ValueError(f"{split} is missing valid clips for: {sorted(missing_labels)}")

train_groups = {rows[i]["group"] for i in split_indices["train"] if rows[i]["storage"] == "expert"}
val_groups = {rows[i]["group"] for i in split_indices["val"] if rows[i]["storage"] == "expert"}
assert train_groups.isdisjoint(val_groups)
assert all(rows[i].get("is_augmented", "").lower() != "true"
           for split in ("val", "test") for i in split_indices[split])

with (OUTPUT_DIR / "splits.csv").open("w", encoding="utf-8-sig", newline="") as stream:
    columns = ["split", "source", "video_id", "sign_id", "group", "rejected_reason"]
    writer = csv.DictWriter(stream, fieldnames=columns)
    writer.writeheader()
    for i, row in enumerate(rows):
        writer.writerow({**{col: row[col] for col in columns[:-1]},
                         "rejected_reason": rejected_by_index.get(i, "")})

print(f"คลาส {len(labels)} | คลิปที่ตัดออก {len(rejected)}")
for split, indices in split_indices.items():
    print(split, len(indices), dict(Counter(rows[i]["source"] for i in indices)))

## 3. เทรน BiLSTM และเลือก checkpoint

โมเดลอ่าน landmark ไปข้างหน้าและย้อนกลับภายในคลิป ใช้น้ำหนัก loss ชดเชยคลาสที่มีตัวอย่างน้อย เลือก checkpoint จาก **validation macro‑F1** พร้อมลด learning rate และหยุดเมื่อผล validation คงที่ ชุด test จะใช้หลังเลือก checkpoint และ threshold แล้ว

In [ ]:
import random
import time

import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cpu":
    print("กำลังรันบน CPU หากต้องการเทรนชุดเต็มเร็วขึ้นให้เลือก GPU ใน Colab")
else:
    print("GPU:", torch.cuda.get_device_name(0))


class ClipDataset(Dataset):
    def __init__(self, data, targets, indices):
        self.data = data
        self.targets = targets
        self.indices = indices

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, index):
        i = self.indices[index]
        return torch.from_numpy(self.data[i]), self.targets[i]


class SignLSTM(nn.Module):
    def __init__(self, input_size, classes, hidden=128):
        super().__init__()
        self.projection = nn.Sequential(nn.Linear(input_size, hidden), nn.LayerNorm(hidden), nn.GELU())
        self.lstm = nn.LSTM(
            hidden, hidden, num_layers=2, bidirectional=True, batch_first=True, dropout=0.25,
        )
        self.head = nn.Sequential(
            nn.LayerNorm(2 * hidden), nn.Dropout(0.30), nn.Linear(2 * hidden, classes),
        )

    def forward(self, landmark_sequence):
        _, (hidden, _) = self.lstm(self.projection(landmark_sequence))
        return self.head(torch.cat((hidden[-2], hidden[-1]), dim=1))


def confusion(truth, pred, classes):
    return np.bincount(classes * truth + pred, minlength=classes**2).reshape(classes, classes)


def class_scores(matrix):
    correct = matrix.diagonal()
    support, predicted = matrix.sum(axis=1), matrix.sum(axis=0)
    precision = correct / np.maximum(predicted, 1)
    recall = correct / np.maximum(support, 1)
    f1 = 2 * correct / np.maximum(support + predicted, 1)
    return precision, recall, f1, support


def metrics(matrix):
    precision, recall, f1, support = class_scores(matrix)
    return {"accuracy": float(matrix.trace() / matrix.sum()),
            "macro_f1": float(f1[support > 0].mean()),
            "macro_precision": float(precision[support > 0].mean()),
            "macro_recall": float(recall[support > 0].mean()),
            "weighted_f1": float(np.sum(f1 * support) / support.sum()),
            "evaluated_classes": int(np.count_nonzero(support)),
            "clips": int(matrix.sum())}


@torch.inference_mode()
def collect_predictions(model, loader):
    model.eval()
    truth, scores = [], []
    for x, y in loader:
        logits = model(x.to(device, non_blocking=True))
        truth.append(y.numpy())
        scores.append(logits.softmax(dim=1).cpu().numpy())
    return np.concatenate(truth), np.concatenate(scores)


loaders = {
    split: DataLoader(
        ClipDataset(features, targets, indices), batch_size=BATCH_SIZE,
        shuffle=split == "train", pin_memory=device.type == "cuda",
        generator=torch.Generator().manual_seed(SEED),
    )
    for split, indices in split_indices.items()
}
model = SignLSTM(preprocess.INPUT_FEATURES, len(labels)).to(device)
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
counts = np.bincount(targets[split_indices["train"]], minlength=len(labels))
weights = 1.0 / np.sqrt(counts)
weights = torch.tensor(weights / weights.mean(), dtype=torch.float32, device=device)
loss_function = nn.CrossEntropyLoss(weight=weights)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.0001)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=3)

best_score, stale_epochs = -1.0, 0
checkpoint = OUTPUT_DIR / "best_weights.pt"
history = []
train_eval_loader = DataLoader(
    ClipDataset(features, targets, split_indices["train"]),
    batch_size=BATCH_SIZE, shuffle=False, pin_memory=device.type == "cuda",
)

def evaluation_loss(truth, probabilities):
    return float(-np.log(np.clip(probabilities[np.arange(len(truth)), truth], 1e-12, 1)).mean())
for epoch in range(1, EPOCHS + 1):
    start = time.perf_counter()
    model.train()
    loss_sum, weight_sum = torch.zeros((), device=device), torch.zeros((), device=device)
    rate = optimizer.param_groups[0]["lr"]
    for x, y in tqdm(loaders["train"], desc=f"Epoch {epoch}/{EPOCHS}", leave=False):
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        loss = loss_function(model(x), y)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        batch_weight = weights[y].sum()
        loss_sum += loss.detach() * batch_weight
        weight_sum += batch_weight

    train_loss = (loss_sum / weight_sum).item()
    if not math.isfinite(train_loss):
        raise RuntimeError("Training loss is not finite")
    train_truth, train_scores = collect_predictions(model, train_eval_loader)
    training = metrics(confusion(train_truth, train_scores.argmax(axis=1), len(labels)))
    truth, scores = collect_predictions(model, loaders["val"])
    validation = metrics(confusion(truth, scores.argmax(axis=1), len(labels)))
    score = validation["macro_f1"]
    scheduler.step(score)

    history.append({
        "epoch": epoch, "train_loss": train_loss,
        "train_eval_loss": evaluation_loss(train_truth, train_scores),
        "val_loss": evaluation_loss(truth, scores),
        **{f"train_{key}": training[key] for key in
           ("accuracy", "macro_precision", "macro_recall", "macro_f1", "weighted_f1")},
        **{f"val_{key}": validation[key] for key in
           ("accuracy", "macro_precision", "macro_recall", "macro_f1", "weighted_f1")},
        "learning_rate": rate, "seconds": time.perf_counter() - start,
    })
    with (OUTPUT_DIR / "history.csv").open("w", encoding="utf-8", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=history[0].keys())
        writer.writeheader()
        writer.writerows(history)

    print(f"epoch {epoch:02d}  loss {train_loss:.4f}  val accuracy {validation['accuracy']:.4f}"
          f"  val macro-F1 {score:.4f}")
    if score > best_score:
        best_score, best_epoch, stale_epochs = score, epoch, 0
        temp = checkpoint.with_suffix(".partial")
        torch.save(model.state_dict(), temp)
        os.replace(temp, checkpoint)
    else:
        stale_epochs += 1
        if stale_epochs >= PATIENCE:
            print("Early stopping")
            break

model.load_state_dict(torch.load(checkpoint, map_location=device, weights_only=True))
print(f"Best epoch: {best_epoch} | validation macro-F1: {best_score:.4f}")

## 4. วัดผลกับคลิปทดสอบ

เลือก threshold จาก validation เพื่อส่งคำทายที่ความมั่นใจต่ำไปเป็น `null_act` จากนั้นรายงานผล test ทั้งก่อนและหลังใช้ threshold แยกผลของ `user_sign` ออกจากคลิปที่เพิ่มเอง ค่า confidence เป็นคะแนน softmax ของโมเดล การวัด `user_sign` ครอบคลุมผู้เซ็นหนึ่งคน ให้เพิ่มชุดทดสอบจากผู้เล่นหลายคนเมื่อต้องการประเมินการใช้งานในเกม

In [ ]:
null_id = label_index["null_act"]
val_truth, val_scores = collect_predictions(model, loaders["val"])

def decide(scores, threshold):
    return np.where(scores.max(axis=1) < threshold, null_id, scores.argmax(axis=1))

thresholds = np.linspace(0, 0.95, 20)
threshold = float(max(thresholds, key=lambda value: metrics(confusion(
    val_truth, decide(val_scores, value), len(labels),
))["macro_f1"]))
test_truth, test_scores = collect_predictions(model, loaders["test"])
np.savez_compressed(
    OUTPUT_DIR / "evaluation_predictions.npz",
    val_truth=val_truth, val_scores=val_scores,
    test_truth=test_truth, test_scores=test_scores,
    test_subset=np.array(["tsl51_user" if rows[i]["storage"] == "user" else "project_extra"
                          for i in split_indices["test"]]),
)

def evaluate_subset(name, mask):
    truth, scores = test_truth[mask], test_scores[mask]
    matrix = confusion(truth, decide(scores, threshold), len(labels))
    raw_matrix = confusion(truth, scores.argmax(axis=1), len(labels))
    precision, recall, f1, support = class_scores(matrix)
    with (OUTPUT_DIR / f"{name}_per_class.csv").open("w", encoding="utf-8-sig", newline="") as stream:
        writer = csv.writer(stream)
        writer.writerow(["sign_id", "thai", "support", "precision", "recall", "f1"])
        for i, label in enumerate(labels):
            writer.writerow([label, thai_names[label], support[i], precision[i], recall[i], f1[i]])
    with (OUTPUT_DIR / f"{name}_confusion.csv").open("w", encoding="utf-8-sig", newline="") as stream:
        writer = csv.writer(stream)
        writer.writerow(["actual/predicted", *labels])
        writer.writerows([label, *counts] for label, counts in zip(labels, matrix))
    return {"argmax": metrics(raw_matrix), "threshold": metrics(matrix)}


tsl51_mask = np.array([rows[i]["storage"] == "user" for i in split_indices["test"]])
results = {"tsl51_user": evaluate_subset("tsl51_user", tsl51_mask)}
if (~tsl51_mask).any():
    results["project_extra"] = evaluate_subset("project_extra", ~tsl51_mask)

summary = {
    "best_epoch": best_epoch,
    "confidence_threshold": threshold,
    "validation_argmax": metrics(confusion(val_truth, val_scores.argmax(axis=1), len(labels))),
    "validation_threshold": metrics(confusion(val_truth, decide(val_scores, threshold), len(labels))),
    "test": results,
}
(OUTPUT_DIR / "metrics.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
print(json.dumps(summary, indent=2))

## 5. ส่งออกและทำนาย

แตก ZIP แล้วใช้ `from tsl_inference import SignPredictor` จากโฟลเดอร์โมเดล เรียก `SignPredictor(".").predict_csv("clip.csv")` สำหรับคลิปหนึ่งท่า โดยติดตั้ง PyTorch และ NumPy ในสภาพแวดล้อมที่ใช้ทำนาย เรียก `predict_landmarks(coordinates, time_ms)` เมื่อต้องการส่ง array รูป `[จำนวนเฟรม, 54, 3]` หรือ `[จำนวนเฟรม, 162]` ตามลำดับคอลัมน์ใน `metadata.json`

เกมต้องส่งช่วงเฟรมของท่าหนึ่งท่าที่ตัดมาแล้ว และส่งพิกัดแบบเดียวกับ TSL‑51 โดยไม่กลับซ้ายขวาจากภาพตัวอย่างกล้อง ค่า `null_act` หมายถึงโมเดลเลือกคลาสพักหรือปฏิเสธคำทายจาก threshold

In [ ]:
%%writefile /content/tsl_inference.py
import json
from pathlib import Path

import torch

from tsl_preprocess import landmarks_to_tensor, read_landmark_csv


class SignPredictor:
    def __init__(self, model_dir):
        model_dir = Path(model_dir)
        self.metadata = json.loads((model_dir / "metadata.json").read_text(encoding="utf-8"))
        self.model = torch.jit.load(str(model_dir / "model.torchscript.pt"), map_location="cpu").eval()

    @torch.inference_mode()
    def _predict(self, sequence):
        probabilities = self.model(torch.from_numpy(sequence).unsqueeze(0)).softmax(dim=1)[0]
        confidence, index = probabilities.max(dim=0)
        index, confidence = index.item(), confidence.item()
        labels = self.metadata["classes"]
        accepted = confidence >= self.metadata["confidence_threshold"]
        chosen = labels[index] if accepted else labels[self.metadata["null_index"]]
        return {
            "sign_id": chosen["sign_id"], "thai": chosen["thai"],
            "raw_sign_id": labels[index]["sign_id"], "raw_confidence": confidence,
            "rejected_low_confidence": not accepted,
        }

    def predict_csv(self, path):
        with Path(path).open(encoding="utf-8-sig", newline="") as stream:
            return self._predict(read_landmark_csv(stream, self.metadata["sequence_frames"]))

    def predict_landmarks(self, coordinates, time_ms=None):
        return self._predict(landmarks_to_tensor(coordinates, self.metadata["sequence_frames"], time_ms))

In [ ]:
import copy

artifact_metadata = {
    "dataset": REPO,
    "revision": REVISION if DOWNLOAD_TSL51 != "local" else "local",
    "dataset_card": f"https://huggingface.co/datasets/{REPO}/blob/{REVISION}/README.md",
    "dataset_license": "CC BY-NC-SA 4.0. See dataset card for third-party terms.",
    "expert_web_scraped_used": INCLUDE_WEB_SCRAPED,
    "extra_manifest_used": EXTRA_MANIFEST is not None,
    "seed": SEED,
    "training": {"epochs": EPOCHS, "batch_size": BATCH_SIZE, "learning_rate": LEARNING_RATE,
                 "patience": PATIENCE, "validation_fraction": VAL_FRACTION},
    "versions": {"torch": str(torch.__version__), "numpy": np.__version__},
    "sequence_frames": FRAMES,
    "input_features": preprocess.INPUT_FEATURES,
    "coordinate_columns": preprocess.COORDINATE_COLUMNS,
    "feature_layout": "162 normalized xyz values followed by 54 valid-landmark masks",
    "output_type": "class logits",
    "classes": [{"index": i, "sign_id": label, "thai": thai_names[label]}
                for i, label in enumerate(labels)],
    "null_index": null_id,
    "confidence_threshold": threshold,
}
(OUTPUT_DIR / "metadata.json").write_text(
    json.dumps(artifact_metadata, indent=2, ensure_ascii=False), encoding="utf-8",
)

archive_files = [
    "model.torchscript.pt", "best_weights.pt", "metadata.json", "metrics.json",
    "tsl_inference.py", "tsl_preprocess.py", "splits.csv", "history.csv",
    *[f"{name}_{kind}.csv" for name in results for kind in ("per_class", "confusion")],
]
shutil.copyfile(preprocess.__file__, OUTPUT_DIR / "tsl_preprocess.py")
shutil.copyfile("/content/tsl_inference.py", OUTPUT_DIR / "tsl_inference.py")

cpu_model = copy.deepcopy(model).cpu().eval()
example_indices = split_indices["test"][:2]
example = torch.from_numpy(np.asarray(features[example_indices]))
with torch.inference_mode():
    exported = torch.jit.trace(cpu_model, example[:1])
    exported.save(str(OUTPUT_DIR / "model.torchscript.pt"))
    reloaded = torch.jit.load(str(OUTPUT_DIR / "model.torchscript.pt")).eval()
    torch.testing.assert_close(reloaded(example), cpu_model(example))

import tsl_inference
tsl_inference = importlib.reload(tsl_inference)
predictor = tsl_inference.SignPredictor(OUTPUT_DIR)
example_row = next(rows[i] for i in split_indices["test"] if rows[i]["storage"] == "user")
prediction = predictor.predict_csv(TSL51_DIR / example_row["landmark_path"])
print("ตัวอย่างคลิปทดสอบ:", example_row["landmark_path"])
print("คำจริง:", example_row["sign_id"], "| คำทำนาย:", prediction)

from zipfile import ZIP_DEFLATED

bundle_path = OUTPUT_DIR / f"{RUN_NAME}_bundle.zip"
with ZipFile(bundle_path.with_suffix(".partial"), "w", compression=ZIP_DEFLATED) as bundle:
    for name in archive_files:
        bundle.write(OUTPUT_DIR / name, arcname=name)
os.replace(bundle_path.with_suffix(".partial"), bundle_path)
print(f"ZIP พร้อมดาวน์โหลด: {bundle_path}")

## กราฟตรวจโมเดลและประสิทธิภาพ

เซลล์ต่อไปนี้วาดกราฟหลังเทรน พร้อมบันทึก PNG, PDF และตาราง CSV ใน `OUTPUT_DIR/plots` และรวมใน ZIP

- Training/validation loss, accuracy, macro precision, recall, F1, weighted F1
- Learning rate, weighted optimization loss, เวลาแต่ละ epoch
- Accuracy/precision/recall/F1 แบบ macro, weighted, micro พร้อม MCC และ Cohen's kappa ใน CSV
- Confusion matrix ทั้งจำนวนคลิปและสัดส่วนต่อแถว
- Precision/recall/F1 และจำนวนคลิปแต่ละคลาส แบ่งหลายหน้าให้เห็นครบทุกคลาส
- คู่คำที่ทายสับสนบ่อย, ROC/Precision-Recall ทุกคลาสแบบ one-vs-rest พร้อม AUC/AP และกราฟรวม
- Histogram ความมั่นใจ, reliability diagram, ECE, NLL และ multiclass Brier score ใน JSON
- ความแม่นยำเมื่อเก็บเฉพาะคลิปความมั่นใจสูง, top-k accuracy และ threshold บน validation
- Latency ของ forward pass ขนาด batch=1 พร้อม p50/p95/p99

Confusion Matrix ใช้คำไทยจริง พร้อมฟอนต์ไทยอัตโนมัติ คำที่มีหลายรูปท่าใช้ sign_id เพื่อระบุรูปท่าให้ชัดเจน กราฟรายคลาสอื่นยังใช้รหัส C00, C01 ฯลฯ และมี `class_mapping.csv` เทียบชื่อไทยและ `sign_id` เพื่อให้เปิดบนเครื่องที่ไม่มีฟอนต์ไทยได้ กราฟ Precision-Recall ใช้ AP เป็นค่ารวม

**การอ่านผล**

- กราฟ train/validation วัดใน evaluation mode หลังจบ epoch ด้วย checkpoint เดียวกัน Loss ที่เทียบกันเป็น cross-entropy เฉลี่ยต่อคลิป ส่วน weighted optimization loss แสดงแยก กิจกรรมนี้เพิ่มเวลาเทรนจากการทำนายชุด train ทุก epoch
- ROC/PR และ calibration ใช้ softmax ก่อน threshold กราฟ confusion และคะแนนรายคลาสใช้ threshold ที่เลือกจาก validation โดยคะแนนต่ำกว่าเกณฑ์ถูกส่งเป็น `null_act`
- Macro ในรายงาน test เฉลี่ยครบคลาสของโมเดล คลาสที่ไม่มีตัวอย่างใน subset ให้ดู support=0 ควบคู่กัน ROC/PR รายคลาสจะแสดงเฉพาะคลาสที่มีทั้งตัวอย่างบวกและลบ
- Coverage เทียบกับ accuracy ใช้ดูพฤติกรรมบน test เกณฑ์ที่ใช้จริงยังมาจาก validation
- Latency วัดบนอุปกรณ์ Colab ด้วย input ที่อยู่ในหน่วยความจำอุปกรณ์แล้ว รวมการรอ GPU ทำงานเสร็จ เป็นเวลาของโมเดลต่อคลิป 48 เฟรม การจับกล้อง สกัด landmark และโอนข้อมูลมีเวลาเพิ่มเติม
- ถ้าใช้ `history.csv` จากรุ่นเก่า จะวาดเฉพาะค่าที่เคยบันทึกไว้ กราฟค่าต่อ epoch ที่เพิ่มใหม่ต้องมีการเทรนด้วยเซลล์ที่อัปเดตนี้

อ้างอิง [scikit-learn metrics](https://scikit-learn.org/stable/modules/model_evaluation.html) และ [PyTorch CUDA timing](https://docs.pytorch.org/docs/stable/notes/cuda.html)

สำหรับ ZIP โมเดลรุ่นเดิม สคริปต์อ่าน history.csv และ confusion CSV เพื่อวาดกราฟที่มีข้อมูลแล้วได้ทันที เมื่อต้องการ ROC/PR และความมั่นใจ ให้รันเซลล์ evaluate และ export ที่อัปเดตใน runtime ที่ยังมี model กับ loaders หรือรันการประเมินใหม่ด้วยข้อมูลเดิม

กราฟ training_validation_loss_accuracy.png แสดง Loss และ Accuracy พร้อม legend Training และ Validation ถ้า history รุ่นเดิมขาดคอลัมน์ จะระบุค่าที่ขาดบนกราฟ ค่า Training/Validation ครบทั้งสองกราฟจะได้เมื่อเทรนด้วย Notebook รุ่นที่บันทึกคอลัมน์ครบ


In [ ]:
%%writefile /content/plot_tsl_metrics.py
"""Plot TSL evaluation artifacts. Run: python plot_tsl_metrics.py MODEL_DIRECTORY."""

import argparse
import csv
import json
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib import font_manager
import numpy as np
from sklearn.metrics import (
    accuracy_score, auc, average_precision_score, cohen_kappa_score,
    confusion_matrix, f1_score, matthews_corrcoef,
    precision_recall_curve, precision_recall_fscore_support, roc_curve,
)


def configure_thai_font(directory):
    for family in ("Tahoma", "Leelawadee UI", "Sarabun", "Noto Sans Thai", "Garuda", "Loma"):
        try:
            font_path = font_manager.findfont(family, fallback_to_default=False)
            break
        except ValueError:
            continue
    else:
        # Colab may need a Thai font. Cache the Google Fonts file beside the plots.
        from urllib.request import urlopen

        directory.mkdir(parents=True, exist_ok=True)
        font_path = directory / "Sarabun-Regular.ttf"
        if not font_path.is_file():
            url = "https://raw.githubusercontent.com/google/fonts/main/ofl/sarabun/Sarabun-Regular.ttf"
            with urlopen(url, timeout=30) as response:
                data = response.read()
            temporary = font_path.with_suffix(".part")
            temporary.write_bytes(data)
            font_manager.FontProperties(fname=str(temporary)).get_name()
            temporary.replace(font_path)
    font_manager.fontManager.addfont(str(font_path))
    plt.rcParams["font.family"] = font_manager.FontProperties(fname=str(font_path)).get_name()
    plt.rcParams["axes.unicode_minus"] = False


def save_figure(fig, directory, name, show):
    from tempfile import NamedTemporaryFile

    fig.tight_layout()
    for extension in ("png", "pdf"):
        destination = directory / f"{name}.{extension}"
        with NamedTemporaryFile(prefix=f"{name}_", suffix=f".{extension}", dir=directory, delete=False) as stream:
            temporary = Path(stream.name)
        try:
            fig.savefig(temporary, format=extension, dpi=180, bbox_inches="tight")
            try:
                temporary.replace(destination)
            except PermissionError:
                print(f"File is open or locked. Saved a new copy to {temporary}")
        except Exception:
            temporary.unlink(missing_ok=True)
            raise
    if show:
        plt.show()
    plt.close(fig)


def write_table(path, rows):
    with path.open("w", encoding="utf-8-sig", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)


def decision(probabilities, threshold, null_index):
    return np.where(probabilities.max(axis=1) < threshold, null_index, probabilities.argmax(axis=1))


def validate_predictions(truth, scores, classes):
    truth, scores = np.asarray(truth), np.asarray(scores)
    if truth.ndim != 1 or not np.issubdtype(truth.dtype, np.integer) or not len(truth):
        raise ValueError("Targets must be a nonempty integer vector")
    if scores.shape != (len(truth), classes) or not np.isfinite(scores).all():
        raise ValueError("Scores must be finite probabilities with shape [samples, classes]")
    if truth.min() < 0 or truth.max() >= classes:
        raise ValueError("Target index outside the class mapping")
    if (scores < 0).any() or (scores > 1).any() or not np.allclose(scores.sum(axis=1), 1, atol=1e-4):
        raise ValueError("Expected softmax probabilities summing to one")


def plot_history(history_path, directory, show):
    directory.mkdir(parents=True, exist_ok=True)
    with history_path.open(encoding="utf-8-sig", newline="") as stream:
        history = list(csv.DictReader(stream))
    if not history:
        raise ValueError("history.csv is empty")
    epochs = np.array([float(row["epoch"]) for row in history])
    train_loss_key = "train_eval_loss" if "train_eval_loss" in history[0] else "train_loss"

    def draw_pair(axis, title, columns):
        missing = []
        for label, column in zip(("Training", "Validation"), columns):
            if column in history[0]:
                axis.plot(epochs, [float(row[column]) for row in history],
                          label=label, linewidth=2, marker=".")
            else:
                missing.append(label)
        axis.set(title=title, xlabel="Epoch", ylabel=title)
        axis.grid(alpha=0.2)
        if axis.lines:
            axis.legend()
        if missing:
            axis.text(0.03, 0.03, f"{' and '.join(missing)} values were not recorded",
                      transform=axis.transAxes, fontsize=9,
                      bbox={"facecolor": "white", "alpha": 0.85, "edgecolor": "none"})

    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
    draw_pair(axes[0], "Loss", (train_loss_key, "val_loss"))
    draw_pair(axes[1], "Accuracy", ("train_accuracy", "val_accuracy"))
    axes[1].set_ylim(0, 1.05)
    if train_loss_key == "train_loss":
        fig.suptitle("Training loss uses the recorded weighted optimization objective", fontsize=10)
    save_figure(fig, directory, "training_validation_loss_accuracy", show)

    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    specifications = [
        ("Cross-entropy (evaluation mode)", ("train_eval_loss", "val_loss")),
        ("Accuracy", ("train_accuracy", "val_accuracy")),
        ("Macro precision", ("train_macro_precision", "val_macro_precision")),
        ("Macro recall", ("train_macro_recall", "val_macro_recall")),
        ("Macro F1", ("train_macro_f1", "val_macro_f1")),
        ("Weighted F1", ("train_weighted_f1", "val_weighted_f1")),
    ]
    for axis, (title, columns) in zip(axes.flat, specifications):
        draw_pair(axis, title, columns)
    save_figure(fig, directory, "learning_curves", show)

    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    for axis, key, title in zip(axes, ("train_loss", "learning_rate", "seconds"),
                                ("Weighted optimization loss", "Learning rate", "Seconds per epoch")):
        if key in history[0]:
            axis.plot(epochs, [float(row[key]) for row in history], marker=".")
        axis.set(title=title, xlabel="Epoch")
        axis.grid(alpha=0.2)
    save_figure(fig, directory, "training_efficiency", show)


def plot_classification(truth, predicted, classes, threshold, directory, show=False, raw=None):
    directory.mkdir(parents=True, exist_ok=True)
    configure_thai_font(directory.parent / "fonts")
    total = len(classes)
    indexes = np.arange(total)
    codes = [f"C{i:02d}" for i in indexes]
    words = [item["thai"] or item["sign_id"] for item in classes]
    word_counts = {word: words.count(word) for word in set(words)}
    tick_labels = [item["sign_id"] if word_counts[word] > 1 else word
                   for word, item in zip(words, classes)]
    write_table(directory / "class_mapping.csv", [
        {"code": codes[i], "sign_id": item["sign_id"], "thai": item["thai"]}
        for i, item in enumerate(classes)
    ])
    rows = []
    decisions = [("Saved threshold decisions", predicted)] if raw is None else [
        ("Argmax", raw), ("Threshold to null", predicted)]
    for name, pred in decisions:
        row = {"decision": name, "accuracy": accuracy_score(truth, pred)}
        for average in ("macro", "weighted", "micro"):
            p, r, f, _ = precision_recall_fscore_support(
                truth, pred, labels=indexes, average=average, zero_division=0,
            )
            row.update({f"{average}_precision": p, f"{average}_recall": r, f"{average}_f1": f})
        row["mcc"] = matthews_corrcoef(truth, pred)
        row["kappa"] = (cohen_kappa_score(truth, pred, labels=indexes)
                        if len(np.unique(np.concatenate([truth, pred]))) > 1 else None)
        rows.append(row)
    write_table(directory / "summary_metrics.csv", rows)
    keys = ["accuracy", "macro_precision", "macro_recall", "macro_f1", "weighted_f1", "micro_f1"]
    fig, axis = plt.subplots(figsize=(10, 4))
    x = np.arange(len(keys))
    for i, row in enumerate(rows):
        axis.bar(x + (i - 0.5) * 0.36, [row[k] for k in keys], 0.36, label=row["decision"])
    axis.set(xticks=x, xticklabels=keys, ylim=(0, 1.05), title="Test classification metrics")
    axis.tick_params(axis="x", rotation=15)
    axis.legend()
    save_figure(fig, directory, "summary_metrics", show)

    fig, axis = plt.subplots(figsize=(8, 4))
    for i, row in enumerate(rows):
        axis.bar(np.arange(2) + (i - 0.5) * 0.36,
                 [row["mcc"], np.nan if row["kappa"] is None else row["kappa"]],
                 0.36, label=row["decision"])
    axis.axhline(0, color="gray", linewidth=0.8)
    axis.set(xticks=[0, 1], xticklabels=["MCC", "Cohen's kappa"], ylim=(-1.05, 1.05),
             title="Agreement and correlation")
    axis.legend()
    save_figure(fig, directory, "agreement_metrics", show)

    cm = confusion_matrix(truth, predicted, labels=indexes)
    support = cm.sum(axis=1)
    normalized = cm / np.maximum(support[:, None], 1)
    for name, matrix in (("counts", cm), ("row_normalized", normalized)):
        fig, axis = plt.subplots(figsize=(max(10, total * 0.32), max(9, total * 0.32)))
        image = axis.imshow(matrix, cmap="Blues", vmin=0, vmax=1 if name == "row_normalized" else None)
        axis.set(xticks=indexes, yticks=indexes, xticklabels=tick_labels, yticklabels=tick_labels,
                 xlabel="Predicted", ylabel="Actual", title=f"Confusion matrix ({name}, threshold={threshold:.2f})")
        axis.tick_params(axis="x", rotation=90, labelsize=9)
        axis.tick_params(axis="y", labelsize=9)
        if total <= 15:
            for (r, c), value in np.ndenumerate(matrix):
                axis.text(c, r, f"{value:.2f}" if name == "row_normalized" else str(value),
                          ha="center", va="center", fontsize=8)
        fig.colorbar(image, ax=axis)
        save_figure(fig, directory, f"confusion_{name}", show)

    p, r, f, support = precision_recall_fscore_support(truth, predicted, labels=indexes, zero_division=0)
    write_table(directory / "per_class.csv", [
        {"code": codes[i], "support": int(support[i]), "precision": p[i], "recall": r[i], "f1": f[i]}
        for i in indexes
    ])
    for start in range(0, total, 20):
        stop = min(start + 20, total)
        fig, axes = plt.subplots(1, 2, figsize=(13, max(4, (stop - start) * 0.35)))
        values = np.arange(stop - start)
        for offset, (name, metric) in enumerate((("Precision", p), ("Recall", r), ("F1", f))):
            axes[0].barh(values + (offset - 1) * 0.23, metric[start:stop], 0.23, label=name)
        axes[0].set(yticks=values, yticklabels=codes[start:stop], xlim=(0, 1.05), title="Per-class scores")
        axes[0].legend()
        axes[1].barh(values, support[start:stop])
        axes[1].set(yticks=values, yticklabels=codes[start:stop], xlabel="Test clips", title="Class support")
        save_figure(fig, directory, f"per_class_{start:02d}", show)

    off_diagonal = cm.copy()
    np.fill_diagonal(off_diagonal, 0)
    pairs = np.argwhere(off_diagonal > 0)
    if len(pairs):
        pairs = sorted(pairs, key=lambda pair: off_diagonal[tuple(pair)], reverse=True)[:20]
        fig, axis = plt.subplots(figsize=(9, 6))
        axis.barh([f"{codes[a]} -> {codes[b]}" for a, b in pairs], [cm[a, b] for a, b in pairs])
        axis.invert_yaxis()
        axis.set(title="Most frequent confusions", xlabel="Clips", ylabel="Actual -> Predicted")
        save_figure(fig, directory, "error_pairs", show)

    return rows


def plot_evaluation(truth, probabilities, classes, threshold, null_index, directory, show=False):
    validate_predictions(truth, probabilities, len(classes))
    directory.mkdir(parents=True, exist_ok=True)
    total = len(classes)
    indexes = np.arange(total)
    codes = [f"C{i:02d}" for i in indexes]
    raw = probabilities.argmax(axis=1)
    predicted = decision(probabilities, threshold, null_index)
    confidence = probabilities.max(axis=1)
    correct = raw == truth
    one_hot = (truth[:, None] == indexes).astype(int)
    rows = plot_classification(truth, predicted, classes, threshold, directory, show, raw)
    support = np.bincount(truth, minlength=total)

    valid_classes = [i for i in indexes if 0 < one_hot[:, i].sum() < len(truth)]
    ranking_rows = []
    for start in range(0, len(valid_classes), 12):
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        for i in valid_classes[start:start + 12]:
            fp, tp, _ = roc_curve(one_hot[:, i], probabilities[:, i])
            precision, recall, _ = precision_recall_curve(one_hot[:, i], probabilities[:, i])
            roc_auc = auc(fp, tp)
            ap = average_precision_score(one_hot[:, i], probabilities[:, i])
            axes[0].plot(fp, tp, label=f"{codes[i]} AUC={roc_auc:.3f}")
            axes[1].plot(recall, precision, label=f"{codes[i]} AP={ap:.3f}")
            ranking_rows.append({"code": codes[i], "roc_auc_ovr": roc_auc, "average_precision": ap})
        axes[0].plot([0, 1], [0, 1], "k--", alpha=0.3)
        axes[0].set(xlabel="False positive rate", ylabel="True positive rate", title="One-vs-rest ROC")
        axes[1].set(xlabel="Recall", ylabel="Precision", title="One-vs-rest Precision-Recall")
        for axis in axes:
            axis.set_xlim(0, 1)
            axis.set_ylim(0, 1.03)
            axis.legend(fontsize=7)
        save_figure(fig, directory, f"roc_pr_classes_{start:02d}", show)
    if ranking_rows:
        write_table(directory / "ranking_metrics.csv", ranking_rows)
    # Micro curves include all model outputs, also for subsets missing some classes.
    fp, tp, _ = roc_curve(one_hot.ravel(), probabilities.ravel())
    precision, recall, _ = precision_recall_curve(one_hot.ravel(), probabilities.ravel())
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(fp, tp, label=f"Micro AUC={auc(fp, tp):.3f}")
    if valid_classes:
        grid = np.linspace(0, 1, 1001)
        macro_tp = []
        for i in valid_classes:
            fpr, tpr, _ = roc_curve(one_hot[:, i], probabilities[:, i])
            macro_tp.append(np.interp(grid, fpr, tpr))
        axes[0].plot(grid, np.mean(macro_tp, axis=0), label="Macro ROC (defined classes)")
    axes[0].plot([0, 1], [0, 1], "k--", alpha=0.3)
    axes[0].set(xlabel="False positive rate", ylabel="True positive rate", title="Aggregate ROC")
    axes[1].plot(recall, precision, label=f"Micro AP={average_precision_score(one_hot.ravel(), probabilities.ravel()):.3f}")
    axes[1].axhline(1 / total, color="gray", linestyle="--", label="Micro prevalence")
    axes[1].set(xlabel="Recall", ylabel="Precision", title="Aggregate Precision-Recall")
    for axis in axes:
        axis.legend()
    save_figure(fig, directory, "roc_pr_aggregate", show)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    bins = np.linspace(0, 1, 11)
    for name, mask in (("Correct", correct), ("Incorrect", ~correct)):
        if mask.any():
            axes[0].hist(confidence[mask], bins=bins, alpha=0.6, label=name)
    axes[0].set(xlabel="Top softmax score", ylabel="Clips", title="Argmax confidence distribution")
    axes[0].legend()
    bin_id = np.minimum((confidence * 10).astype(int), 9)
    bin_rows = []
    for i in range(10):
        mask = bin_id == i
        if mask.any():
            bin_rows.append({"bin": i, "count": int(mask.sum()),
                             "confidence": float(confidence[mask].mean()),
                             "accuracy": float(correct[mask].mean())})
    ece = sum(abs(row["accuracy"] - row["confidence"]) * row["count"] / len(truth) for row in bin_rows)
    axes[1].plot([0, 1], [0, 1], "k--", label="Perfect calibration")
    axes[1].scatter([row["confidence"] for row in bin_rows], [row["accuracy"] for row in bin_rows],
                    s=[20 + 200 * row["count"] / len(truth) for row in bin_rows])
    axes[1].set(xlim=(0, 1), ylim=(0, 1), xlabel="Mean top score", ylabel="Observed accuracy",
                title=f"Reliability diagram (10 equal bins), ECE={ece:.3f}")
    save_figure(fig, directory, "confidence_calibration", show)
    write_table(directory / "calibration_bins.csv", bin_rows)
    nll = -np.log(np.clip(probabilities[np.arange(len(truth)), truth], 1e-12, 1)).mean()
    probability_metrics = {"nll": float(nll), "multiclass_brier_sum": float(np.square(probabilities - one_hot).sum(axis=1).mean()),
                           "top_label_ece_10_bins": float(ece), "absent_test_classes": int((support == 0).sum())}
    (directory / "probability_metrics.json").write_text(json.dumps(probability_metrics, indent=2), encoding="utf-8")
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    for axis, key, title in zip(axes, ("nll", "multiclass_brier_sum", "top_label_ece_10_bins"),
                                ("Negative log-likelihood", "Multiclass Brier (sum)", "Top-label ECE")):
        value = probability_metrics[key]
        axis.bar([title], [value])
        axis.set(title="Lower is better", ylim=(0, max(value * 1.25, 0.05)))
        axis.text(0, value, f"{value:.4f}", ha="center", va="bottom")
    save_figure(fig, directory, "probability_metrics", show)

    thresholds = np.linspace(0, 1, 51)
    coverage, accuracy = [], []
    for value in thresholds:
        retained = confidence >= value
        coverage.append(retained.mean())
        accuracy.append(correct[retained].mean() if retained.any() else np.nan)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(coverage, accuracy, marker=".")
    axes[0].set(xlabel="Retained fraction (includes predicted null)", ylabel="Accuracy on retained clips",
                title="Coverage vs accuracy (test diagnostic)", ylim=(0, 1.05))
    order = np.argsort(-probabilities, axis=1, kind="stable")
    ks = sorted({1, min(3, total), min(5, total), min(10, total)})
    topk = [(order[:, :k] == truth[:, None]).any(axis=1).mean() for k in ks]
    axes[1].bar([str(k) for k in ks], topk)
    axes[1].set(xlabel="k", ylabel="Accuracy", title="Top-k accuracy (raw scores)", ylim=(0, 1.05))
    save_figure(fig, directory, "coverage_topk", show)
    return rows


def generate_report(model_dir, show=False):
    model_dir = Path(model_dir)
    metadata = json.loads((model_dir / "metadata.json").read_text(encoding="utf-8"))
    directory = model_dir / "plots"
    directory.mkdir(exist_ok=True)
    if (model_dir / "history.csv").is_file():
        plot_history(model_dir / "history.csv", directory, show)
    classes = metadata["classes"]
    threshold, null_index = metadata["confidence_threshold"], metadata["null_index"]
    prediction_file = model_dir / "evaluation_predictions.npz"
    if not prediction_file.is_file():
        found = False
        for path in sorted(model_dir.glob("*_confusion.csv")):
            with path.open(encoding="utf-8-sig", newline="") as stream:
                reader = csv.reader(stream)
                header, records = next(reader), list(reader)
            labels = [item["sign_id"] for item in classes]
            if header[1:] != labels or [row[0] for row in records] != labels:
                raise ValueError(f"Class order mismatch in {path}")
            matrix = np.array([[int(value) for value in row[1:]] for row in records])
            if matrix.shape != (len(classes), len(classes)) or (matrix < 0).any() or not matrix.sum():
                raise ValueError(f"Invalid confusion matrix in {path}")
            actual, predicted = np.indices(matrix.shape)
            # Recover aggregate decision counts without inventing softmax scores.
            truth = np.repeat(actual.ravel(), matrix.ravel())
            pred = np.repeat(predicted.ravel(), matrix.ravel())
            plot_classification(truth, pred, classes, threshold,
                                directory / path.stem.removesuffix("_confusion"), show)
            found = True
        if not found:
            raise FileNotFoundError("Need evaluation_predictions.npz or *_confusion.csv")
        print("Existing bundle plotted from saved threshold decisions and recorded history.")
        print("ROC/PR, confidence and top-k plots require per-clip softmax scores from the updated evaluation cell.")
        print(f"Saved PNG, PDF, CSV reports to {directory}")
        return directory
    with np.load(prediction_file, allow_pickle=False) as saved:
        for subset in np.unique(saved["test_subset"]):
            mask = saved["test_subset"] == subset
            plot_evaluation(saved["test_truth"][mask], saved["test_scores"][mask], classes,
                            threshold, null_index, directory / str(subset), show)
        truth, scores = saved["val_truth"], saved["val_scores"]
        validate_predictions(truth, scores, len(classes))
        thresholds = np.linspace(0, 1, 101)
        fig, axis = plt.subplots(figsize=(9, 4))
        for name, metric in (("Accuracy", accuracy_score), ("Macro F1", None)):
            values = []
            for value in thresholds:
                pred = decision(scores, value, null_index)
                values.append(metric(truth, pred) if metric else f1_score(
                    truth, pred, labels=np.arange(len(classes)), average="macro", zero_division=0))
            axis.plot(thresholds, values, label=name)
        axis.axvline(threshold, linestyle="--", color="black", label=f"Saved threshold={threshold:.2f}")
        axis.set(xlabel="Confidence threshold mapped to null", ylabel="Score", title="Validation threshold diagnostic")
        axis.legend()
        save_figure(fig, directory, "validation_threshold", show)
    benchmark_file = model_dir / "inference_benchmark.json"
    if benchmark_file.is_file():
        benchmark = json.loads(benchmark_file.read_text(encoding="utf-8"))
        fig, axes = plt.subplots(1, 2, figsize=(11, 4))
        axes[0].hist(benchmark["latency_ms"], bins=20)
        axes[0].set(xlabel="ms per batch-1 forward pass", ylabel="Runs", title=f"Inference latency on {benchmark['device']}")
        axes[1].bar(["p50", "p95", "p99"], np.percentile(benchmark["latency_ms"], [50, 95, 99]))
        axes[1].set(ylabel="Milliseconds", title="Model latency percentiles")
        save_figure(fig, directory, "inference_latency", show)
    print(f"Saved PNG, PDF, CSV reports to {directory}")
    return directory


if __name__ == "__main__":
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("model_dir", type=Path)
    parser.add_argument("--show", action="store_true")
    args = parser.parse_args()
    generate_report(args.model_dir, show=args.show)


In [ ]:
import importlib.util

for module, package in (("matplotlib", "matplotlib"), ("sklearn", "scikit-learn")):
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

# Warm up first, and synchronize GPU work before recording each measurement.
model.eval()
benchmark_input = torch.from_numpy(np.asarray(features[split_indices["test"][:1]])).to(device)
latencies = []
with torch.inference_mode():
    for _ in range(20):
        model(benchmark_input)
    if device.type == "cuda":
        torch.cuda.synchronize(device)
    for _ in range(100):
        start = time.perf_counter()
        model(benchmark_input)
        if device.type == "cuda":
            torch.cuda.synchronize(device)
        latencies.append((time.perf_counter() - start) * 1000)

(OUTPUT_DIR / "inference_benchmark.json").write_text(json.dumps({
    "device": torch.cuda.get_device_name(device) if device.type == "cuda" else "CPU",
    "batch_size": 1, "frames": FRAMES, "warmup_runs": 20, "latency_ms": latencies,
    "torch_version": str(torch.__version__), "cpu_threads": torch.get_num_threads(),
    "scope": "Model forward pass on device-resident preprocessed input",
}, indent=2), encoding="utf-8")

import plot_tsl_metrics
plot_tsl_metrics = importlib.reload(plot_tsl_metrics)
plot_dir = plot_tsl_metrics.generate_report(OUTPUT_DIR, show=True)
shutil.copyfile("/content/plot_tsl_metrics.py", OUTPUT_DIR / "plot_tsl_metrics.py")

# Rebuild the bundle with evaluation inputs and generated plots.
extra_files = [OUTPUT_DIR / name for name in
               ("evaluation_predictions.npz", "inference_benchmark.json", "plot_tsl_metrics.py")]
with ZipFile(bundle_path.with_suffix(".partial"), "w", compression=ZIP_DEFLATED) as bundle:
    for path in [*[OUTPUT_DIR / name for name in archive_files], *extra_files, *sorted(plot_dir.rglob("*"))]:
        if path.is_file():
            bundle.write(path, arcname=path.relative_to(OUTPUT_DIR).as_posix())
os.replace(bundle_path.with_suffix(".partial"), bundle_path)
print("Model and plot ZIP:", bundle_path)


In [ ]:
# ตั้ง True แล้วรันเซลล์นี้เพื่อดาวน์โหลด ZIP จาก Colab
DOWNLOAD_BUNDLE = False
if DOWNLOAD_BUNDLE:
    from google.colab import files
    files.download(str(bundle_path))

## 6. ดาวน์โหลดชุดเสริมจากรายงาน

เซลล์นี้เลือกเปิดทีละแหล่ง ใช้เก็บข้อมูลสำหรับงานต่อยอด โดยการเทรนด้านบนอ่าน TSL‑51 และชุดที่ระบุใน `EXTRA_MANIFEST` เท่านั้น ชุด ThaiSignVis มีวิดีโอจำนวนมาก จึงควรเลือกไฟล์หรือโฟลเดอร์จากรายการ Kaggle ในการทดลองครั้งแรก หาก Kaggle ขอสิทธิ์ ให้เข้าเว็บไซต์ยอมรับเงื่อนไขและเพิ่ม `KAGGLE_API_TOKEN` ผ่านเมนู Secrets ของ Colab [วิธีใช้ Kaggle Hub](https://github.com/Kaggle/kagglehub)

สำหรับ One‑Stage‑TFS ให้กด **Download All** ที่ [Mendeley Data](https://data.mendeley.com/datasets/rknd3wbz42/1) แล้วเก็บ ZIP ไว้ใน Drive ส่วน TSL‑ONE‑S แบบ MP4 ให้ขอผ่าน [ฟอร์มโครงการ](https://tsl-one.github.io/) แหล่งที่ต้องติดต่อผู้วิจัยดูได้จากตารางในหัวข้อ 1

In [ ]:
DOWNLOAD_TSL_ONE_KEYPOINTS = False
DOWNLOAD_ATFS_REPOSITORY = False
DOWNLOAD_THAISIGNVIS = False
THAISIGNVIS_PATH = None   # ระบุ path ตามหน้า Kaggle หรือใช้ None เพื่อดาวน์โหลดทั้งชุด

def download_archive(url, destination, signature):
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.is_file():
        with destination.open("rb") as stream:
            if stream.read(len(signature)) == signature:
                return destination
    partial = destination.with_suffix(".part")
    subprocess.check_call([
        "curl", "--fail", "--location", "--retry", "4", "--retry-all-errors",
        "--connect-timeout", "30", "--output", str(partial), url,
    ])
    with partial.open("rb") as stream:
        if stream.read(len(signature)) != signature:
            raise ValueError(f"Unexpected archive format from {url}")
    os.replace(partial, destination)
    return destination


if DOWNLOAD_TSL_ONE_KEYPOINTS:
    print(download_archive(
        "https://raw.githubusercontent.com/TSL-ONE/TSL-ONE.github.io/"
        "62ff4924873fbb20b8691bc36591dfa72c0b4f9a/data/TSL-ONE-Pose.7z",
        WORK_DIR / "public_datasets/TSL-ONE-Pose.7z",
        b"7z\xbc\xaf\x27\x1c",
    ))

if DOWNLOAD_ATFS_REPOSITORY:
    print(download_archive(
        "https://codeload.github.com/beebrain/ATFS_Thai_Finger_Spelling/zip/refs/heads/master",
        WORK_DIR / "public_datasets/ATFS_repository.zip", b"PK",
    ))

if DOWNLOAD_THAISIGNVIS:
    if importlib.util.find_spec("kagglehub") is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "kagglehub"])
    import kagglehub

    print(kagglehub.dataset_download(
        "thanawuttimpitak/thaisignvis", path=THAISIGNVIS_PATH,
        output_dir=str(WORK_DIR / "public_datasets/ThaiSignVis"),
    ))